# Random Survival Forest (RSF) - Training Pipeline

Questo notebook definisce la pipeline di addestramento per il modello Random Survival Forest applicato al dataset Head & Neck Cancer. Il processo include la gestione avanzata dei dati mancanti e l'ottimizzazione degli iperparametri.

---

### Fase 1: Caricamento e Preprocessing
Il primo passo consiste nel caricare i dati e prepararli prima della divisione in set. Le operazioni principali sono:
* **Gestione dei Missing nativi:** I valori codificati come `-1` (utilizzati nello step precedente per i dati mancanti categoriali e del TNM) vengono convertiti in `NaN` per poter essere processati correttamente dagli algoritmi di imputazione. I valori numerici mancanti legati all'esposizione (alcol, fumo) sono già preservati come `NaN` nativi dalla pipeline di estrazione, pronti per il MICE.

### Fase 2: Train / Test Split
Il dataset viene diviso in un set di addestramento (75%) e uno di test (25%).
Lo split è rigorosamente stratificato sulla variabile evento (`vital_status_DEAD`) per assicurare che la proporzione di eventi avversi (decessi) sia bilanciata tra i due set.

### Fase 3: Imputazione dei Dati Mancanti (MICE)
Per risolvere i `NaN` presenti nel dataset, applichiamo l'algoritmo MICE (Multiple Imputation by Chained Equations tramite `IterativeImputer`).
* **Prevenzione del Data Leakage:** L'imputer viene "addestrato" (fit) esclusivamente sui dati di training. Successivamente, le regole apprese vengono applicate (transform) sia al training che al test set. Questo garantisce che nessuna informazione del test set influenzi il modello.

### Fase 4: Ottimizzazione degli Iperparametri (RSF)
Per massimizzare le performance del Random Survival Forest, utilizziamo una ricerca randomizzata (`RandomizedSearchCV`).
Vengono testate 50 combinazioni di iperparametri (es. profondità degli alberi, numero di stimatori) utilizzando una Cross-Validation a 5 fold (5-fold CV). Questo approccio offre un eccellente bilanciamento tra esplorazione dello spazio dei parametri e tempi di calcolo.

### Fase 5: Valutazione Rapida e Salvataggio
Viene calcolato il C-index sui dati di training e test per una validazione istantanea della bontà del modello (sanity check). Infine, l'imputer e il modello ottimizzato vengono salvati in formato `.pkl` all'interno della cartella `../models/` per essere riutilizzati nello step di valutazione.

In [28]:
import os, warnings
import numpy as np
import pandas as pd
import joblib
from sksurv.util import Surv
from sksurv.ensemble import RandomSurvivalForest
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

warnings.filterwarnings("ignore")

# ==============================================================================
# CONFIGURAZIONE E PATHS
# ==============================================================================
DATA_PATH = "../data/dataset.csv"

MODEL_DIR    = "../models"
MODEL_PATH   = os.path.join(MODEL_DIR, "rsf_model.pkl")
IMPUTER_PATH = os.path.join(MODEL_DIR, "mice_imputer.pkl")
os.makedirs(MODEL_DIR, exist_ok=True)

TIME_COL     = "follow_ups.days_to_follow_up"
EVENT_COL    = "demographic.vital_status_DEAD"
RANDOM_STATE = 20
TEST_SIZE    = 0.25

# ==============================================================================
# FASE 1: CARICAMENTO E PREPROCESSING
# ==============================================================================
df = pd.read_csv(DATA_PATH)

df = df.replace(-1, np.nan)

X = df.drop(columns=[TIME_COL, EVENT_COL])
y = Surv.from_arrays(event=df[EVENT_COL].astype(bool), time=df[TIME_COL])

# ==============================================================================
# FASE 2: SPLIT TRAIN / TEST
# ==============================================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=df[EVENT_COL],
)

# ==============================================================================
# FASE 3: IMPUTAZIONE MICE
# ==============================================================================
imputer = IterativeImputer(random_state=RANDOM_STATE, max_iter=10)
X_train_imp = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns)
X_test_imp = pd.DataFrame(imputer.transform(X_test), columns=X_test.columns)

joblib.dump(imputer, IMPUTER_PATH)

# ==============================================================================
# FASE 4: OTTIMIZZAZIONE RSF E TRAINING
# ==============================================================================
param_distributions = {
    "n_estimators":      [200, 500, 1000],
    "max_depth":         [3, 5, 10, None],
    "min_samples_split": [5, 10, 20, 30],
    "min_samples_leaf":  [5, 10, 20, 40],
    "max_features":      ["sqrt", "log2", None],
}

rsf_base = RandomSurvivalForest(random_state=RANDOM_STATE, n_jobs=-1)

search = RandomizedSearchCV(
    rsf_base,
    param_distributions,
    n_iter=50,
    cv=5,
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbose=0,
)
search.fit(X_train_imp, y_train)
best_model = search.best_estimator_

# ==============================================================================
# FASE 5: VALUTAZIONE E SALVATAGGIO
# ==============================================================================
ci_train = best_model.score(X_train_imp, y_train)
ci_test  = best_model.score(X_test_imp,  y_test)

joblib.dump(best_model, MODEL_PATH)

print("=" * 62)
print("  TRAINING COMPLETATO - RANDOM SURVIVAL FOREST")
print("=" * 62)
print(f"  CV C-index (best) : {search.best_score_:.4f}")
print(f"  C-index TRAIN     : {ci_train:.4f}")
print(f"  C-index TEST      : {ci_test:.4f}")
print(f"  Artefatti salvati in: {MODEL_DIR}/")
print("=" * 62)

  TRAINING COMPLETATO - RANDOM SURVIVAL FOREST
  CV C-index (best) : 0.7299
  C-index TRAIN     : 0.8511
  C-index TEST      : 0.7164
  Artefatti salvati in: ../models/


# Random Survival Forest (RSF) - Evaluation & Explainable AI

Questo notebook è dedicato alla valutazione clinica e tecnica del modello RSF addestrato. Oltre a misurare le performance predittive, il notebook utilizza tecniche di Explainable AI (XAI) per rendere trasparenti le logiche decisionali del modello sia a livello globale che sul singolo paziente.

---

### Fase 1: Ricostruzione dell'Ambiente e Preprocessing
Per garantire una valutazione corretta, l'ambiente di test deve essere identico a quello del training:
* Il dataset viene ricaricato e subisce la stessa pulizia di base (es. sostituzione dei -1 usati come segnaposto categoriale con `NaN`). Le variabili di esposizione arrivano già con i `NaN` nativi, pronti per l'imputazione.
* Viene effettuato lo split con lo stesso `random_state` per isolare l'esatto Test Set usato in fase di addestramento.
* Vengono caricati l'imputer MICE e il modello RSF salvati in precedenza, applicando la trasformazione ai dati.

### Fase 2: Metriche Globali di Sopravvivenza
Il modello viene valutato attraverso tre metriche standard per la Survival Analysis:
* **C-index:** Valuta la capacità di discriminazione (il modello assegna un rischio maggiore a chi ha un tempo di sopravvivenza minore?).
* **Integrated Brier Score (IBS):** Misura l'accuratezza e la calibrazione delle probabilità nel tempo. Un valore più basso indica una migliore calibrazione rispetto al benchmark naive.
* **Time-dependent AUC:** Calcola l'Area Sotto la Curva a specifici orizzonti temporali di interesse clinico (1, 3 e 5 anni) per capire come si evolve il potere predittivo.

### Fase 3: Stratificazione del Rischio (Validazione Clinica)
I pazienti del test set vengono divisi in due gruppi ("Alto Rischio" e "Basso Rischio") utilizzando il valore mediano del Risk Score predetto come soglia.
Le curve di sopravvivenza stimate per le due coorti vengono tracciate per verificare se il modello riesce a separare nettamente le traiettorie cliniche dei due gruppi.

### Fase 4: Explainable AI Globale (Permutation Feature Importance)
Per comprendere quali variabili sono più rilevanti per l'intero modello, applichiamo la Permutation Feature Importance.
L'algoritmo rimescola casualmente i valori di una variabile alla volta: se la feature è cruciale per il modello, il C-index crollerà significativamente. Questo processo viene ripetuto 10 volte per garantire stabilità statistica ai risultati.

### Fase 5: Explainable AI Locale (SHAP)
Per ottenere interpretabilità a livello del singolo paziente, impieghiamo la libreria SHAP (SHapley Additive exPlanations):
* **SHAP Summary Plot:** Fornisce una panoramica globale di come le singole variabili (es. età, stadio del tumore) spingano il rischio verso l'alto o verso il basso.
* **SHAP Waterfall Plot:** Esamina casi clinici specifici (il paziente a rischio massimo e quello a rischio minimo nel test set), mostrando il contributo esatto di ciascuna feature clinica nella determinazione del loro punteggio finale.

In [29]:
import os, warnings
import numpy as np
import pandas as pd
import joblib
import shap
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance
from sksurv.util import Surv
from sksurv.metrics import (integrated_brier_score,
                            cumulative_dynamic_auc)

warnings.filterwarnings("ignore")

# ==============================================================================
# CONFIGURAZIONE E PATHS
# ==============================================================================
RESULTS_DIR  = "results"
DATA_PATH    = "../data/dataset.csv"
MODEL_DIR    = "../models"
MODEL_PATH   = os.path.join(MODEL_DIR, "rsf_model.pkl")
IMPUTER_PATH = os.path.join(MODEL_DIR, "mice_imputer.pkl")

os.makedirs(RESULTS_DIR, exist_ok=True)

for _f, _name in [(MODEL_PATH, "rsf_model.pkl"), (IMPUTER_PATH, "mice_imputer.pkl")]:
    if not os.path.exists(_f):
        raise FileNotFoundError(f"{_name} non trovato in {MODEL_DIR}.\nEsegui prima il training.")

TIME_COL     = "follow_ups.days_to_follow_up"
EVENT_COL    = "demographic.vital_status_DEAD"
RANDOM_STATE = 20
TEST_SIZE    = 0.25

# ==============================================================================
# 1. CARICAMENTO DATI, IMPUTAZIONE E MODELLO
# ==============================================================================
print("=" * 62)
print("  EVALUATION & XAI - RANDOM SURVIVAL FOREST (RSF)")
print("=" * 62)

df = pd.read_csv(DATA_PATH)
df = df.replace(-1, np.nan)

X_all = df.drop(columns=[TIME_COL, EVENT_COL])
t_all = df[TIME_COL].values
e_all = df[EVENT_COL].values.astype(int)

X_train_raw, X_test_raw, t_train, t_test, e_train, e_test = train_test_split(
    X_all, t_all, e_all,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=e_all,
)

imputer = joblib.load(IMPUTER_PATH)
rsf     = joblib.load(MODEL_PATH)

X_train = pd.DataFrame(imputer.transform(X_train_raw), columns=X_train_raw.columns)
X_test  = pd.DataFrame(imputer.transform(X_test_raw),  columns=X_test_raw.columns)

y_train = Surv.from_arrays(event=e_train.astype(bool), time=t_train)
y_test  = Surv.from_arrays(event=e_test.astype(bool),  time=t_test)

risk_scores = rsf.predict(X_test)
print(f"[+] Modello caricato | Test set: {len(X_test)} pz | {e_test.sum()} eventi\n")

# ==============================================================================
# FASE 1: METRICHE GLOBALI
# ==============================================================================
print("=" * 62)
print("  FASE 1: METRICHE GLOBALI")
print("=" * 62)

c_index = rsf.score(X_test, y_test)

lb        = np.percentile(t_test, 10)
ub        = np.percentile(t_test, 90)
ibs_times = np.linspace(lb, ub, 20)

surv_funcs = rsf.predict_survival_function(X_test)
preds_ibs  = np.clip(np.vstack([fn(ibs_times) for fn in surv_funcs]), 1e-6, 1 - 1e-6)
ibs = integrated_brier_score(y_train, y_test, preds_ibs, ibs_times)

tempi_giorni = [365, 1095, 1825]
etichette    = ["1 anno (365 gg)", "3 anni (1095 gg)", "5 anni (1825 gg)"]
min_t, max_t = t_test.min(), t_test.max()

tempi_validi, label_validi = [], []
for t_g, lbl in zip(tempi_giorni, etichette):
    if min_t < t_g < max_t:
        tempi_validi.append(t_g)
        label_validi.append(lbl)

auc_vals, mean_auc = None, None
if tempi_validi:
    auc_vals, mean_auc = cumulative_dynamic_auc(y_train, y_test, risk_scores, np.array(tempi_validi))

print(f"  C-index (test set) : {c_index:.4f}")
print(f"  IBS                : {ibs:.4f} (baseline ~0.25, ↓ meglio)")
if mean_auc:
    print(f"  Mean AUC           : {mean_auc:.4f}")
    for lbl, auc in zip(label_validi, auc_vals):
        print(f"    - AUC a {lbl} : {auc:.4f}")

# ==============================================================================
# FASE 2: STRATIFICAZIONE DEL RISCHIO
# ==============================================================================
print("\n" + "=" * 62)
print("  FASE 2: STRATIFICAZIONE CLINICA (Alto vs Basso Rischio)")
print("=" * 62)

median_risk = np.median(risk_scores)
mask_high   = risk_scores > median_risk
mask_low    = ~mask_high
plot_times  = np.linspace(lb, ub, 100)

surv_high = rsf.predict_survival_function(X_test[mask_high])
surv_low  = rsf.predict_survival_function(X_test[mask_low])
mean_surv_high = np.mean(np.vstack([fn(plot_times) for fn in surv_high]), axis=0)
mean_surv_low  = np.mean(np.vstack([fn(plot_times) for fn in surv_low]),  axis=0)

fig1, ax1 = plt.subplots(figsize=(10, 6))
ax1.step(plot_times, mean_surv_low,  where="post", label="Basso Rischio", color="#2ca02c", lw=2.5)
ax1.step(plot_times, mean_surv_high, where="post", label="Alto Rischio", color="#d62728", lw=2.5)
ax1.fill_between(plot_times, mean_surv_low, mean_surv_high, alpha=0.08, color="grey", step="post")
ax1.set_title("Curve di Sopravvivenza (RSF): Alto vs Basso Rischio", fontsize=13, fontweight="bold")
ax1.set_ylabel("Probabilità di Sopravvivenza", fontsize=11)
ax1.set_xlabel("Giorni di Follow-up", fontsize=11)
ax1.set_ylim(0, 1.02)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.35, linestyle="--")
plt.tight_layout()

plt.savefig(os.path.join(RESULTS_DIR, "rsf_survival_curves.png"), dpi=150)
plt.close()
print(f"  [+] Grafico curve salvato in: {RESULTS_DIR}")

# ==============================================================================
# FASE 3: XAI GLOBALE (PERMUTATION FEATURE IMPORTANCE)
# ==============================================================================
print("\n" + "=" * 62)
print("  FASE 3: PERMUTATION FEATURE IMPORTANCE")
print("=" * 62)
print("  Calcolo in corso (10 ripetizioni)...")

perm = permutation_importance(
    rsf, X_test, y_test,
    n_repeats=10,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

imp_mean = np.maximum(perm.importances_mean, 0)
imp_pct  = 100.0 * imp_mean / (imp_mean.sum() + 1e-12)

df_imp = pd.DataFrame({
    "Feature":        X_test.columns,
    "Importanza (%)": imp_pct,
    "Std":            perm.importances_std,
}).sort_values("Importanza (%)", ascending=False)

fig2, ax2 = plt.subplots(figsize=(9, 7))
ax2.barh(df_imp["Feature"][::-1], df_imp["Importanza (%)"][::-1], color="#1f77b4", xerr=df_imp["Std"][::-1], capsize=3)
ax2.set_xlabel("Importanza relativa (%)", fontsize=11)
ax2.set_title("Permutation Feature Importance – RSF", fontsize=12, fontweight="bold")
ax2.grid(axis="x", alpha=0.35, linestyle="--")
plt.tight_layout()

plt.savefig(os.path.join(RESULTS_DIR, "rsf_permutation_importance.png"), dpi=150)
plt.close()
print(f"  [+] Grafico Permutation salvato in: {RESULTS_DIR}")

# ==============================================================================
# FASE 4: XAI LOCALE (SHAP)
# ==============================================================================
print("\n" + "=" * 62)
print("  FASE 4: XAI LOCALE (SHAP)")
print("=" * 62)

def _predict_risk_shap(X_raw):
    if isinstance(X_raw, pd.DataFrame):
        return rsf.predict(X_raw)
    return rsf.predict(pd.DataFrame(X_raw, columns=X_test.columns))

background     = shap.sample(X_train, 100, random_state=RANDOM_STATE)
explainer_shap = shap.Explainer(_predict_risk_shap, background)

print("  Calcolo SHAP su tutto il test set...")
shap_values_all = explainer_shap(X_test)

plt.figure(figsize=(10, 7))
shap.plots.beeswarm(shap_values_all, max_display=15, show=False)
plt.title("SHAP Summary – RSF (Contributo al Risk Score)", fontsize=12, fontweight="bold")
plt.tight_layout()

plt.savefig(os.path.join(RESULTS_DIR, "rsf_shap_summary.png"), dpi=150, bbox_inches="tight")
plt.close()
print("  [+] SHAP Summary plot salvato")

idx_alto  = int(np.argmax(risk_scores))
idx_basso = int(np.argmin(risk_scores))

for label, idx in [("ALTO RISCHIO", idx_alto), ("BASSO RISCHIO", idx_basso)]:
    plt.figure(figsize=(14, 7))
    try:
        shap.plots.waterfall(shap_values_all[idx], max_display=13, show=False, matplotlib=True)
    except TypeError:
        shap.plots.waterfall(shap_values_all[idx], max_display=13, show=False)

    ax_wf = plt.gca()
    plt.title(f"SHAP Waterfall – RSF – Paziente {label}", fontsize=12, fontweight="bold", pad=12)
    ax_wf.tick_params(axis="y", labelsize=9)
    for text in ax_wf.texts:
        if "f(x)" not in text.get_text() and "base value" not in text.get_text():
            text.set_fontsize(8)
    plt.subplots_adjust(left=0.42, right=0.95, top=0.90, bottom=0.10)
    ax_wf.grid(axis="x", alpha=0.3, linestyle="--")

    fname = f"rsf_shap_waterfall_{label.lower().replace(' ', '_')}.png"

    plt.savefig(os.path.join(RESULTS_DIR, fname), dpi=150, bbox_inches="tight")
    plt.close()

print("  [+] SHAP Waterfall plots salvati per pazienti ad alto e basso rischio")

# ==============================================================================
# RIEPILOGO FINALE
# ==============================================================================
print("\n" + "=" * 62)
print("  RIEPILOGO FINALE – Random Survival Forest")
print("=" * 62)
print(f" Esecuzione completata con successo. Tutti i grafici sono salvati in:\n  {RESULTS_DIR}")

  EVALUATION & XAI - RANDOM SURVIVAL FOREST (RSF)
[+] Modello caricato | Test set: 132 pz | 56 eventi

  FASE 1: METRICHE GLOBALI
  C-index (test set) : 0.7164
  IBS                : 0.1887 (baseline ~0.25, ↓ meglio)
  Mean AUC           : 0.6997
    - AUC a 1 anno (365 gg) : 0.7641
    - AUC a 3 anni (1095 gg) : 0.6895
    - AUC a 5 anni (1825 gg) : 0.6355

  FASE 2: STRATIFICAZIONE CLINICA (Alto vs Basso Rischio)
  [+] Grafico curve salvato in: results

  FASE 3: PERMUTATION FEATURE IMPORTANCE
  Calcolo in corso (10 ripetizioni)...
  [+] Grafico Permutation salvato in: results

  FASE 4: XAI LOCALE (SHAP)
  Calcolo SHAP su tutto il test set...


PermutationExplainer explainer: 133it [23:20, 10.61s/it]                         


  [+] SHAP Summary plot salvato
  [+] SHAP Waterfall plots salvati per pazienti ad alto e basso rischio

  RIEPILOGO FINALE – Random Survival Forest
  ✅ Esecuzione completata con successo. Tutti i grafici sono salvati in:
  results
